# Markov Random Fields and Mean Field Theory

Prepared for Fall 2026. The foreground/background model and the images come from the Gibbs sampling notebook of the Fall 2025 homework (originally by Weichao Qiu).

References: the notes for Lecture 6 (Probabilities on Graphs: Spatial Context) and Lecture 7 (Mean Field Theory: image labelling, the weak membrane and binocular stereo). Gibbs sampling, used in one section below, is explained in that section and in Lecture 8.

If you find a bug in this notebook, please email the TAs or tell us at office hours.

**Before you start.** Run the notebook from the homework folder, so that the folder `data` is next to it. Every figure and number that a question asks for must appear in your PDF: when a question asks you to vary a parameter, copy the relevant cell into the code cell below the question (or write a loop) instead of editing and re-running the cell above, which would overwrite the earlier figure.

---

**The model.** Every pixel $i$ has a binary label $x_i \in \{0, 1\}$ (1 = foreground, 0 = background) and an observed grey level $z_i$. The energy of a labelling is evidence plus context,

$$E(\mathbf{x}) = \sum_i \psi_i(x_i) + w \sum_{(i,j)} [x_i \ne x_j], \qquad \psi_i(x_i) = \frac{(z_i - \mu_{x_i})^2}{2 s^2},$$

where $[x_i \ne x_j]$ is 1 when the two labels differ and 0 when they agree, and the sum runs over the pairs of neighbouring pixels.

The unary term $\psi_i$ says how well the grey level fits the mean $\mu_1$ of the foreground or the mean $\mu_0$ of the background, with noise level $s$. The pairwise term is the Potts model on the four-neighbour grid: every pair of neighbours with different labels costs $w$. The energy defines the Gibbs distribution $P(\mathbf{x} \mid \mathbf{z}) = e^{-E(\mathbf{x})/T} / Z$ at temperature $T$.

**Mean field.** Mean field theory approximates $P$ by a factorised distribution with one belief $q_i = Q(x_i = 1)$ per pixel. Switching pixel $i$ on changes the unary cost by $\Delta\psi_i = \psi_i(1) - \psi_i(0)$ and the expected pairwise cost by $w \sum_{j \in N(i)} (1 - 2 q_j)$, so the update of Lecture 6 reads

$$q_i \leftarrow \sigma\!\left(\frac{-\Delta\psi_i + w \sum_{j \in N(i)} (2 q_j - 1)}{T}\right), \qquad \sigma(u) = \frac{1}{1 + e^{-u}}.$$

It is a stationary point of the mean field free energy (the expected energy plus $T$ times the negative entropy):

$$F(\mathbf{q}) = \sum_i \big[q_i \psi_i(1) + (1 - q_i)\psi_i(0)\big] + w \sum_{(i,j)} \big[q_i(1 - q_j) + q_j(1 - q_i)\big] + T \sum_i \big[q_i \log q_i + (1 - q_i)\log(1 - q_i)\big].$$

## Utilities

In [ ]:
from __future__ import annotations
import os
import time

import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
from PIL import Image

%matplotlib inline

if not os.path.isdir("data/mrf"):
    raise FileNotFoundError("The folder data/mrf was not found. Start Jupyter in the homework folder, "
                            "so that the folder data is next to this notebook.")


def sigmoid(u):
    return 1 / (1 + np.exp(-np.clip(u, -50, 50)))


def neighbour_sum(a: npt.NDArray) -> npt.NDArray:
    """Sum of the four neighbours (up, down, left, right) of every pixel; pixels outside the image count as 0."""
    s = np.zeros_like(a, dtype=float)
    s[1:, :] += a[:-1, :]
    s[:-1, :] += a[1:, :]
    s[:, 1:] += a[:, :-1]
    s[:, :-1] += a[:, 1:]
    return s


def unary_potentials(z, mu0: float, mu1: float, s: float):
    """psi_i(0) and psi_i(1): the cost of calling pixel i background or foreground."""
    return (z - mu0) ** 2 / (2 * s**2), (z - mu1) ** 2 / (2 * s**2)


def energy(x, psi0, psi1, w: float) -> float:
    """E(x) = sum of unary costs + w * (number of neighbouring pairs with different labels)."""
    unary = np.where(x == 1, psi1, psi0).sum()
    disagreements = np.sum(x[:, 1:] != x[:, :-1]) + np.sum(x[1:, :] != x[:-1, :])
    return float(unary + w * disagreements)


def free_energy(q, psi0, psi1, w: float, T: float = 1.0) -> float:
    """Mean field free energy F(q): expected energy + T * negative entropy."""
    qc = np.clip(q, 1e-12, 1 - 1e-12)
    expected_unary = np.sum(q * psi1 + (1 - q) * psi0)
    expected_pairwise = w * (np.sum(q[:, 1:] * (1 - q[:, :-1]) + q[:, :-1] * (1 - q[:, 1:]))
                             + np.sum(q[1:, :] * (1 - q[:-1, :]) + q[:-1, :] * (1 - q[1:, :])))
    negative_entropy = np.sum(qc * np.log(qc) + (1 - qc) * np.log(1 - qc))
    return float(expected_unary + expected_pairwise + T * negative_entropy)


def checkerboard(shape) -> npt.NDArray[np.bool_]:
    """True on the 'black' squares of a checkerboard. No two black pixels are neighbours."""
    yy, xx = np.indices(shape)
    return (yy + xx) % 2 == 0


def mean_field(psi0, psi1, w: float, T: float = 1.0, n_iter: int = 30, schedule: str = "checkerboard", q0=None):
    """Mean field iterations for the binary Potts model.

    schedule = "checkerboard": update all black pixels, then all white pixels (one iteration = both halves)
    schedule = "parallel":     update every pixel at once from the beliefs of the previous iteration

    Returns the beliefs q and a history with the free energy after every iteration and the number
    of pixels whose label (q > 0.5) flipped during the iteration.
    """
    d_psi = psi1 - psi0
    n_nb = neighbour_sum(np.ones_like(d_psi))
    q = sigmoid(-d_psi / T) if q0 is None else q0.astype(float).copy()
    black = checkerboard(q.shape)
    history = {"F": [free_energy(q, psi0, psi1, w, T)], "flips": []}

    def update(q):
        return sigmoid((-d_psi + w * (2 * neighbour_sum(q) - n_nb)) / T)

    for _ in range(n_iter):
        old = q > 0.5
        if schedule == "parallel":
            q = update(q)
        elif schedule == "checkerboard":
            for mask in (black, ~black):
                q = np.where(mask, update(q), q)
        else:
            raise ValueError(schedule)
        history["F"].append(free_energy(q, psi0, psi1, w, T))
        history["flips"].append(int(np.sum((q > 0.5) != old)))
    return q, history


def gibbs_sampler(psi0, psi1, w: float, T: float = 1.0, n_sweeps: int = 300, burn_in: int = 50, seed: int = 0):
    """Gibbs sampling with a checkerboard schedule: all black pixels are resampled given the white ones, then the reverse.

    Returns two arrays: the estimated marginals P(x_i = 1), i.e. the average of the samples after burn-in,
    and the last sample.
    """
    rng = np.random.default_rng(seed)
    d_psi = psi1 - psi0
    n_nb = neighbour_sum(np.ones_like(d_psi))
    black = checkerboard(d_psi.shape)
    x = (rng.random(d_psi.shape) < sigmoid(-d_psi / T)).astype(float)
    total, n = np.zeros_like(d_psi), 0
    for sweep in range(n_sweeps):
        for mask in (black, ~black):
            p_on = sigmoid((-d_psi + w * (2 * neighbour_sum(x) - n_nb)) / T)
            x = np.where(mask, (rng.random(x.shape) < p_on).astype(float), x)
        if sweep >= burn_in:
            total += x
            n += 1
    return total / n, x


def make_synthetic_scene(size: int = 96) -> npt.NDArray[np.int64]:
    """Ground-truth labels: a disk, a rectangle and a thin bar on a background."""
    yy, xx = np.indices((size, size))
    x = np.zeros((size, size), dtype=int)
    x[(yy - 38) ** 2 + (xx - 30) ** 2 < 17**2] = 1
    x[(yy >= 56) & (yy < 86) & (xx >= 52) & (xx < 88)] = 1
    x[(yy >= 12) & (yy < 15) & (xx >= 50) & (xx < 90)] = 1
    return x


def load_image(name: str, max_side: int = 160) -> npt.NDArray[np.float64]:
    """Load data/mrf/<name>.jpg as grey levels in [0, 1], shrunk so that its longer side is at most max_side."""
    image = Image.open(f"data/mrf/{name}.jpg").convert("L")
    scale = max_side / max(image.size)
    if scale < 1:
        image = image.resize((round(image.size[0] * scale), round(image.size[1] * scale)), Image.LANCZOS)
    return np.array(image).astype(float) / 255


def show(images, titles, **kwargs):
    fig, axes = plt.subplots(1, len(images), figsize=(3.2 * len(images), 3.2))
    for ax, image, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(image, cmap="gray", **kwargs)
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    plt.show()

## A synthetic image with known ground truth

The observed grey level is the mean of the true class plus Gaussian noise: $z_i = \mu_{x_i} + \text{noise}$, with $\mu_0 = 0.3$, $\mu_1 = 0.7$ and noise standard deviation $s = 0.25$. With $w = 0$ every pixel is decided on its own evidence.

In [ ]:
truth = make_synthetic_scene()
rng = np.random.default_rng(0)
mu0, mu1, s = 0.3, 0.7, 0.25
z = np.where(truth == 1, mu1, mu0) + rng.normal(scale=s, size=truth.shape)
psi0, psi1 = unary_potentials(z, mu0, mu1, s)

pixelwise = (psi1 < psi0).astype(int)
show([truth, z, pixelwise], ["ground truth", "observed image z", f"pixel by pixel: {100 * np.mean(pixelwise != truth):.1f}% wrong"])

## Mean field segmentation

The beliefs start from the unary evidence alone, $q_i = \sigma(-\Delta\psi_i)$, and every iteration updates the black and then the white pixels of a checkerboard. The labelling is read off the beliefs: $\hat{x}_i = 1$ if $q_i > 0.5$.

In [ ]:
def segment(z, mu0, mu1, s, w, T=1.0, n_iter=30):
    """Mean field segmentation; returns the beliefs, the labelling and the free-energy history."""
    psi0, psi1 = unary_potentials(z, mu0, mu1, s)
    q, history = mean_field(psi0, psi1, w, T=T, n_iter=n_iter)
    return q, (q > 0.5).astype(int), history


w = 1.0   # try other values for Question 9.1
q, labelling, history = segment(z, mu0, mu1, s, w)
show([q, labelling], [f"beliefs q, w = {w:g}", f"labelling: {100 * np.mean(labelling != truth):.1f}% wrong"], vmin=0, vmax=1)

plt.figure(figsize=(4.5, 3))
plt.plot(history["F"], marker="o", ms=3)
plt.xlabel("iteration")
plt.ylabel("free energy F(q)")
plt.show()

### Question 9.1 (3 points)

- Run mean field on the synthetic image with $w \in \{0, 0.25, 0.5, 1, 2, 8\}$ (write a loop in the code cell below). Report the percentage of wrong labels for each value and show the labellings. **(1 point)**
- Explain the effect of $w$ in terms of the two terms of the energy. What happens when $w$ is too small? What goes wrong when $w$ is large (compare the thin bar and the outline of the disk for $w = 2$ and $w = 8$)? **(1 point)**
- Apply the model to one of the real images in `data/mrf` (`cat`, `moon` or `einstein`). Use the histogram of grey levels (cell below) to choose $\mu_0$, $\mu_1$ and $s$, and show the result for $w = 0$ and for one larger value of $w$. **(1 point)**

In [ ]:
image = load_image("cat")        # or "moon", "einstein"
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(image, cmap="gray")
axes[0].axis("off")
axes[1].hist(image.ravel(), bins=50)
axes[1].set_xlabel("grey level")
axes[1].set_ylabel("number of pixels")
fig.tight_layout()
plt.show()

# Your code for Question 9.1, e.g.
# q, labelling, history = segment(image, mu0=..., mu1=..., s=..., w=...)

**Your answers for Question 9.1:**

## Update schedules

The checkerboard schedule updates all black pixels at once and then all white pixels. The parallel schedule updates every pixel at once, using the beliefs of the previous iteration. Both are run below with $w = 2$ from the same initial beliefs: the unary evidence, and an artificial checkerboard pattern.

In [ ]:
w = 2.0
starts = {"unary evidence": None, "checkerboard pattern": np.where(checkerboard(z.shape), 0.9, 0.1)}

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for col, (start_name, q0) in enumerate(starts.items()):
    for schedule, color in (("checkerboard", "tab:green"), ("parallel", "tab:red")):
        q, history = mean_field(psi0, psi1, w, n_iter=40, schedule=schedule, q0=q0)
        axes[0, col].plot(history["F"], color=color, label=schedule)
        axes[1, col].plot(range(1, 41), history["flips"], color=color, label=schedule)
        decreasing = np.all(np.diff(history["F"]) <= 1e-9)
        print(f"start: {start_name:21s} schedule: {schedule:12s} F never increases: {decreasing}   "
              f"last four values of F: {np.round(history['F'][-4:], 1)}   labels flipped in the last iteration: {history['flips'][-1]}")
    axes[0, col].set_title(f"start: {start_name}")
    axes[0, col].set_ylabel("free energy F(q)")
    axes[0, col].set_yscale("log")
    axes[0, col].yaxis.set_major_locator(LogLocator(subs=(1.0, 1.5, 2.0, 3.0, 5.0, 7.0)))
    axes[0, col].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}"))
    axes[0, col].yaxis.set_minor_formatter(NullFormatter())
    axes[1, col].set_ylabel("labels flipped in the iteration")
    axes[1, col].set_xlabel("iteration")
    axes[1, col].set_yscale("symlog")
    axes[1, col].set_ylim(-0.3, 5e4)
    for ax in axes[:, col]:
        ax.legend()
fig.tight_layout()
plt.show()

# The parallel schedule from the checkerboard start: the labels after its last two iterations
q39, _ = mean_field(psi0, psi1, w, n_iter=39, schedule="parallel", q0=starts["checkerboard pattern"])
q40, _ = mean_field(psi0, psi1, w, n_iter=40, schedule="parallel", q0=starts["checkerboard pattern"])
zoom = (slice(40, 52), slice(40, 52))   # a 12 x 12 window, so that single pixels are visible
print("Labels of the parallel run from the checkerboard start, in a 12 x 12 window of the image:")
show([(q39 > 0.5)[zoom], (q40 > 0.5)[zoom]], ["iteration 39", "iteration 40"], interpolation="nearest")

### Question 9.2 (3 points)

- Plot the free energy against the iteration for the two schedules and both starts. Report, for each schedule and each start, whether $F$ ever increases. **(1 point)**
- One schedule is safe and the other is not: explain why, in terms of the beliefs that each update reads. What do you see in the number of labels that flip at every iteration, and what does it tell you about where the parallel update ends up? (The last figure of the cell shows the labels of the parallel run from the checkerboard start after its last two iterations.) **(2 points)**

In [ ]:
# Your code for Question 9.2

**Your answers for Question 9.2:**

## Mean field versus Gibbs sampling

Gibbs sampling draws samples from $P(\mathbf{x} \mid \mathbf{z})$ by resampling every pixel from its conditional distribution given its neighbours; the average of the samples after a burn-in period estimates the true marginals $P(x_i = 1 \mid \mathbf{z})$. Mean field replaces the samples by deterministic beliefs.

In [ ]:
w = 1.0
t0 = time.time()
q_mf, _ = mean_field(psi0, psi1, w, n_iter=30)
time_mf = time.time() - t0

t0 = time.time()
p_gibbs, last_sample = gibbs_sampler(psi0, psi1, w, n_sweeps=300, burn_in=50)
time_gibbs = time.time() - t0

show([q_mf, p_gibbs, np.abs(q_mf - p_gibbs)],
     ["mean field beliefs", "Gibbs estimate of the marginals", "absolute difference"], vmin=0, vmax=1)
for name, p, t in (("mean field", q_mf, time_mf), ("Gibbs sampling", p_gibbs, time_gibbs)):
    uncertain = np.mean((p > 0.1) & (p < 0.9))
    print(f"{name:15s}: {100 * uncertain:4.1f}% of the pixels have a belief between 0.1 and 0.9, "
          f"{100 * np.mean((p > 0.5) != truth):.2f}% wrong labels, {t:.3f} s")

plt.figure(figsize=(5, 3))
plt.hist([q_mf.ravel(), p_gibbs.ravel()], bins=20, label=["mean field", "Gibbs"], log=True)
plt.xlabel("belief / estimated marginal")
plt.ylabel("number of pixels")
plt.legend()
plt.show()

### Question 9.3 (3 points)

- Show the mean field beliefs and the Gibbs estimate of the marginals. Which of the two is more confident? Report the percentage of pixels whose belief lies between 0.1 and 0.9 for each method. **(1 point)**
- Explain the difference in confidence between the two methods, using the form of the approximating distribution $Q$. **(1 point)**
- Compare the running times. Why is mean field faster, and when would you still prefer sampling? **(1 point)**

In [ ]:
# Your code for Question 9.3

**Your answers for Question 9.3:**

## Deterministic annealing

Mean field finds a local minimum of $F$. Here it starts from a bad initial state, $q_i = 0.02$ everywhere ("everything is background"), with a strong pairwise weight $w = 2$. Deterministic annealing starts the same state at a high temperature $T$, where $F$ is smooth, and tracks the minimum while $T$ is lowered to 1.

In [ ]:
w = 2.0
q_bad = np.full(z.shape, 0.02)

q_direct, _ = mean_field(psi0, psi1, w, T=1.0, n_iter=60, q0=q_bad)

q_anneal = q_bad.copy()
temperatures = np.geomspace(6.0, 1.0, 12)
for T in temperatures:
    q_anneal, _ = mean_field(psi0, psi1, w, T=T, n_iter=5, q0=q_anneal)
q_anneal, _ = mean_field(psi0, psi1, w, T=1.0, n_iter=20, q0=q_anneal)

results = {"mean field at T = 1": q_direct, "deterministic annealing": q_anneal}
for name, q in results.items():
    x_hat = (q > 0.5).astype(int)
    print(f"{name:24s}: E(x) = {energy(x_hat, psi0, psi1, w):8.1f}, {100 * np.mean(x_hat != truth):.1f}% wrong labels")
show([(q > 0.5) for q in results.values()], list(results.keys()), vmin=0, vmax=1)

### Question 9.4 (2 points)

Report the energy $E(\hat{\mathbf{x}})$ and the percentage of wrong labels for mean field at $T=1$ and for deterministic annealing, both started from the bad initial state. Explain why mean field at $T = 1$ gets stuck, and why starting at a high temperature helps. **(2 points)**

In [ ]:
# Your code for Question 9.4

**Your answers for Question 9.4:**

## The weak membrane

The weak membrane smooths a noisy signal $\mathbf{z}$ and keeps its discontinuities. Between neighbouring samples $i$ and $j = i + 1$ there is a line process $y_{ij} \in \{0, 1\}$; $y_{ij} = 1$ breaks the membrane. The posterior has the energy

$$\tau \sum_i (x_i - z_i)^2 + A \sum_{i} (x_i - x_{i+1})^2 (1 - y_{i,i+1}) + B \sum_{i} y_{i,i+1}.$$

The EM algorithm of Lecture 7 alternates two steps, starting from $\mathbf{x} = \mathbf{z}$:

- **E-step:** the probability of a break, $b_{ij} = \sigma\big(A (x_i - x_j)^2 - B\big)$;
- **M-step:** the smooth output, $\mathbf{x} = \arg\min_{\mathbf{x}} \big\{ \tau \sum_i (x_i - z_i)^2 + A \sum_i (1 - b_{i,i+1})(x_i - x_{i+1})^2 \big\}$, a linear system.

Minimising the energy over the line process gives each edge the cost $\min(A d^2, B)$, where $d = x_i - x_j$. The EM algorithm sums over the line process instead, which replaces this cost by its smooth version $-\log\big(e^{-A d^2} + e^{-B}\big)$ (Lecture 7).

In [ ]:
def make_signal(n: int = 80, noise: float = 0.06, seed: int = 0):
    """A piecewise smooth signal: a flat level, a slow ramp and another flat level, and a noisy copy."""
    rng = np.random.default_rng(seed)
    x = np.empty(n)
    x[:25] = 0.2
    x[25:55] = np.linspace(0.7, 0.9, 30)
    x[55:] = 0.4
    return x, x + rng.normal(scale=noise, size=n)


def m_step(z, b, tau: float, A: float):
    """Minimise tau * sum (x - z)^2 + A * sum (1 - b) * (x_i - x_{i+1})^2 by solving a linear system."""
    n = len(z)
    M = np.diag(np.full(n, tau, dtype=float))
    weights = A * (1 - b)
    for i in range(n - 1):
        M[i, i] += weights[i]
        M[i + 1, i + 1] += weights[i]
        M[i, i + 1] -= weights[i]
        M[i + 1, i] -= weights[i]
    return np.linalg.solve(M, tau * z)


def weak_membrane_free_energy(x, b, z, tau, A, B):
    bc = np.clip(b, 1e-12, 1 - 1e-12)
    d = np.diff(x)
    return float(tau * np.sum((x - z) ** 2) + A * np.sum((1 - b) * d**2) + B * np.sum(b)
                 + np.sum(bc * np.log(bc) + (1 - bc) * np.log(1 - bc)))


def weak_membrane_em(z, tau: float = 60, A: float = 600, B: float = 6, n_iter: int = 30):
    """EM for the weak membrane; returns x, the break probabilities b and the free energy after every half-step."""
    x = z.copy()
    F = []
    for _ in range(n_iter):
        b = sigmoid(A * np.diff(x) ** 2 - B)          # E-step
        F.append(weak_membrane_free_energy(x, b, z, tau, A, B))
        x = m_step(z, b, tau, A)                      # M-step
        F.append(weak_membrane_free_energy(x, b, z, tau, A, B))
    return x, b, np.array(F)


signal, z1 = make_signal()
x, b, F = weak_membrane_em(z1)
rmse = lambda a: np.sqrt(np.mean((a - signal) ** 2))
print(f"breaks (b > 0.5) between samples {[(int(i), int(i) + 1) for i in np.nonzero(b > 0.5)[0]]}")
print(f"RMSE of the input: {rmse(z1):.4f}, RMSE of the output: {rmse(x):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2), gridspec_kw={"width_ratios": [2, 2, 1.2]})
axes[0].plot(z1, "o", color="0.6", ms=3, label="input z")
axes[0].plot(signal, color="k", lw=1, ls=":", label="noise-free signal")
axes[0].plot(x, color="tab:blue", lw=2, label="weak membrane x")
for i in np.nonzero(b > 0.5)[0]:
    axes[0].axvline(i + 0.5, color="tab:orange", ls="--")
axes[0].set_xlabel("sample i")
axes[0].set_ylabel("value")
axes[0].legend(fontsize=8)
axes[1].bar(np.arange(len(b)) + 0.5, b, color="tab:orange")
axes[1].set_ylabel("break probability $b_{i,i+1}$")
axes[1].set_xlabel("position between samples")
axes[2].plot(F, marker="o", ms=3)
axes[2].set_xlabel("half-step")
axes[2].set_ylabel("free energy")
fig.tight_layout()
plt.show()

# Plain quadratic smoothing: the same M-step with no line processes (b = 0 everywhere)
x_plain = m_step(z1, np.zeros(len(z1) - 1), tau=60, A=600)
print(f"RMSE of plain smoothing (b = 0): {rmse(x_plain):.4f}")

### Question 9.5 (3 points)

- Run the EM algorithm with $B \in \{0.5, 2, 6, 20, 100\}$, keeping $\tau = 60$ and $A = 600$. For each value report the number of breaks ($b_{ij} > 0.5$) and the RMSE against the noise-free signal, and show the outputs for the smallest and the largest $B$. **(1 point)**
- Explain the effect of $B$ with the cost of one edge of the membrane, $\min(A d^2, B)$ for a difference $d = x_i - x_j$. Which value of $B$ finds exactly the two true discontinuities? **(1 point)**
- Plot the output of plain quadratic smoothing (the array `x_plain`, computed at the end of the cell above) and compare it with the weak membrane. Why does the weak membrane do better near the discontinuities? **(1 point)**

In [ ]:
# Your code for Question 9.5

**Your answers for Question 9.5:**